# LinkedIn Job Postings Scraper

Scrapes job postings via the Apify `linkedin-jobs-scraper` actor across a
title x location grid. Requires an Apify API token (see setup below).

**Input:** none (live scrape)
**Output:** `data/mia_postings_raw.csv`

Downstream steps (not yet documented as notebooks — see repo README):
raw -> `mia_postings_final.csv` -> `Final Cleaning.ipynb` -> `mia_postings_final2.csv`
-> `mia_postings_final2_fixed2.csv` -> everything else.

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from apify_client import ApifyClient

In [ ]:
# --- Apify credentials ---
# Set your token as an environment variable before running this notebook —
# never hardcode API tokens in a notebook that gets committed to git.
#   macOS/Linux:  export APIFY_API_TOKEN="your_token_here"
#   Windows:      setx APIFY_API_TOKEN "your_token_here"
API_TOKEN = os.environ.get("APIFY_API_TOKEN")
if not API_TOKEN:
    raise ValueError("Set the APIFY_API_TOKEN environment variable before running this notebook.")

ACTOR_ID = "valig/linkedin-jobs-scraper"
client = ApifyClient(API_TOKEN)
print("Apify client initialized.")

In [ ]:
# Search terms used to query LinkedIn. Note: LinkedIn matches by intent, not
# literal title — only ~5% of postings literally contain the searched phrase
# (see Inferences.md, "Market & segmentation"). Digital Marketing Analyst is
# the exception at 15.6% literal match rate.
TITLES = [
    "Marketing Analyst",
    "Marketing Analytics",
    "Digital Analytics",
    "Marketing Data Analyst",
    "Marketing Insights Analyst",
    "Digital Marketing Analyst",
]

In [ ]:
# Countries covered in the scrape. AI-mention rate varies substantially by
# country (26% France to 58% Canada) — geography is a real filter, not noise.
LOCATIONS = [
    "United States",
    "United Kingdom",
    "Switzerland",
    "Netherlands",
    "China",
    "India",
    "Germany",
    "France",
    "Japan",
    "Canada",
]

In [ ]:
DATE_POSTED = "r2592000"  # Apify actor param: trailing 30 days, in seconds

In [ ]:
FIELDS_TO_KEEP = [
    "title",
    "companyName",
    "location",
    "workType",
    "postedTimeAgo",
    "applicationsCount",
    "url",
    "description",
]

In [ ]:
all_rows = []
total_runs = len(TITLES) * len(LOCATIONS)
run_count = 0

for title in TITLES:
    for location in LOCATIONS:
        run_count += 1
        print(f"[{run_count}/{total_runs}] Scraping: {title} — {location}")
        run_input = {
            "title": title,
            "location": location,
            "datePosted": DATE_POSTED,
            "limit": 150,  # capped to stay within remaining Apify credits
            # worst case: 60 queries x 150 = 9,000 jobs (~$3.60 at $0.4/1K)
            # "experienceLevel": ["1", "2"],  # uncomment to restrict to
            # internship ("1") / entry level ("2") if postings run too senior
        }

        run = client.actor(ACTOR_ID).call(run_input=run_input)
        # Newer apify-client versions return a Run object, not a dict —
        # try attribute access first, fall back to dict-style for older versions
        try:
            dataset_id = run.default_dataset_id
        except AttributeError:
            dataset_id = run["defaultDatasetId"]

        for item in client.dataset(dataset_id).iterate_items():
            row = {field: item.get(field) for field in FIELDS_TO_KEEP}
            row["search_title_used"] = title  # track which query found it
            row["search_location_used"] = location
            all_rows.append(row)

        time.sleep(2)  # small pause between actor calls

In [ ]:
df = pd.DataFrame(all_rows)
print(f"Total rows before dedupe: {len(df)}")

In [ ]:
# Dedup on URL, not title+company+location. An earlier title+company+location
# approach would have wrongly deleted 94 legitimate rows — the same real job
# posted under two different search-term buckets, not true duplication.
# See Inferences.md, "Data integrity" section.
df = df.drop_duplicates(subset="url", keep="first")
print(f"Total rows after dedupe: {len(df)}")

In [ ]:
output_path = Path("data") / "mia_postings_raw.csv"
output_path.parent.mkdir(exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")